In [ ]:
from __future__ import annotations

import operator
import os
import re
from typing import TypedDict,List,Annotated,Literal
from pydantic import BaseModel,Field
from langgraph.graph import StateGraph,START,END
from langgraph.types import Send
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage,HumanMessage
from pathlib import Path

# POSTGRES DATABASE
import psycopg
from langgraph.checkpoint.postgres import PostgresSaver
from psycopg.rows import dict_row
from dotenv import load_dotenv

load_dotenv()

True

In [7]:
def get_database_url():
    database_url = os.getenv("DATABASE_URL")

    if not database_url:
        raise ValueError(
            "DATABASE_URL is missing. Please add your Render PostgreSQL External Database URL to .env"
        )

    if "sslmode=" not in database_url:
        #If a query string already exists (has '?'), append with '&'. Otherwise, start it with '?'.
        separator = "&" if "?" in database_url else "?"
        database_url = f"{database_url}{separator}sslmode=require"

    # ? => marks beginning of query
    # & => separates multiple query parameters.If only one query parameter no need to use &
    # ex: https://example.com?parameter1=value1&parameter2=value2&parameter3=value3
    return database_url

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
if not GROQ_API_KEY:
    raise ValueError("GROQ_API_KEY is missing. Please add it to your .env file.")

In [ ]:
class Task(BaseModel):
    id: int
    title: str
    brief: str = Field(..., description="What to cover")

class Plan(BaseModel):
    blog_title: str
    tasks: List[Task]

class State(TypedDict):
    topic: str
    plan: Plan
    # reducer: results from workers get concatenated automatically
    sections: Annotated[List[str], operator.add]
    final: str

llm = ChatGroq(model="openai/gpt-oss-20b",api_key=GROQ_API_KEY)

def orchestrator(state: State) -> dict:
    # create a simple plan for each section
    plan = llm.with_structured_output(Plan).invoke(
        [SystemMessage(content=("Create a blog plan with 5-7 sub sections for blog on the following topic.")),
        HumanMessage(content=f"Topic: {state['topic']}"),])
    return {"plan": plan}

def fanout(state: State):
    # print(state["plan"])
    # for each task in tasks in "plan" call worker for parallel exectuion
    return [Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
            for task in state["plan"].tasks]

def worker(payload: dict) -> dict:
    # payload contains what we sent
    task = payload["task"]
    topic = payload["topic"]
    plan = payload["plan"]

    blog_title = plan.blog_title
    section_md = llm.invoke(
        [SystemMessage(content="Write one clean Markdown section under 20 to 30 words."),
        HumanMessage(content=(
                    f"Blog: {blog_title}\n"
                    f"Topic: {topic}\n\n"
                    f"Section: {task.title}\n"
                    f"Brief: {task.brief}\n\n"
                    "Write about the section topic and return only the section content in Markdown.")),]
    ).content.strip()
    # print(section_md)
    return {"sections": [section_md]}

def reducer(state: State) -> dict:
    # Receives output from all the worker node and merges it
    title = state["plan"].blog_title
    body = "\n\n".join(state["sections"]).strip()
    final_md = f"# {title}\n\n{body}\n"
    print(final_md)
    # ---- save to file ----
    clean_title = re.sub(r'[^\w\s-]', '', title).strip().lower()
    filename = re.sub(r'[-\s]+', '_', clean_title) + ".md"
    # print(filename)
    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")
    # print(f"File saved to absolute path: {output_path.resolve()}")
    return {"final": final_md}

graph=StateGraph(State)

# add nodes
graph.add_node('orchestrator',orchestrator)
graph.add_node('worker',worker)
graph.add_node('reducer',reducer)

# add edges
graph.add_edge(START,'orchestrator')
graph.add_conditional_edges('orchestrator',fanout,['worker'])
graph.add_edge('worker','reducer')
graph.add_edge('reducer',END)


In [9]:
# =========================
# PostgreSQL Checkpointer
# =========================
DATABASE_URL = get_database_url()

_conn = psycopg.connect(
    DATABASE_URL,
    autocommit=True,
    row_factory=dict_row
)

checkpointer = PostgresSaver(_conn)
checkpointer.setup()

# Note: Whenever you wish to create a connection between agent and Database use checkpointer
# Pass the created checkpointer while compiling the graph
workflow=graph.compile(checkpointer=checkpointer)
# workflow

In [10]:
config = {"configurable": {"thread_id": "test_thread"}}
out = workflow.invoke({"topic": "Write a blog on Bellman Ford's Algorithm", "sections": []}, config=config)

# Bellman-Ford's Algorithm: A Comprehensive Guide

A linked list is a linear collection of nodes, each holding data and a reference to the next node, enabling dynamic memory usage and efficient insertions.

## Types of Linked Lists  
Singly, doubly, and circular lists differ in node links: a single forward link, bidirectional pointers, or wrap‑around. Use cases: simple queues, bidirectional traversal, and circular buffers.

**Core Operations & Algorithms**

Insertion, deletion, traversal, search, and reverse in linked lists—illustrated with concise C++, Java, and Python snippets.

# Performance & Complexity Analysis

Linked lists enable O(1) end insert/delete but O(n) random access; arrays give O(1) access, O(n) insertion, with lower space overhead.

## Advanced Topics & Variants

Skip lists accelerate search, while linked lists underpin efficient queues and stacks. Memory tricks like node pooling and compact pointers reduce overhead.

## Practical Use Cases

Linked lists power OS proc